In [1]:
from functions import query_snowflake_to_df
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score


In [2]:
team = 'LOUCITY'

In [3]:
query = f"""
WITH sampled_users AS (
    SELECT UNIFY_ID
    FROM (
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_MARKETING_ACTIVITIES WHERE UNIFY_ID IS NOT NULL
        UNION
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_TICKET_SALES         WHERE UNIFY_ID IS NOT NULL
        UNION
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_MERCH_TRANSACTIONS   WHERE UNIFY_ID IS NOT NULL
    )
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(UNIFY_ID)) <= 10000
),

deduped_tickets AS (
    SELECT
        ORDER_ID,
        MAX(UNIFY_ID)             AS UNIFY_ID,
        MAX(ORDER_DATE)           AS ORDER_DATE,
        SUM(TOTAL_GROSS_SALES)    AS TOTAL_GROSS_SALES,
        SUM(TOTAL_NET_SALES)      AS TOTAL_NET_SALES,
        SUM(TOTAL_DISCOUNT)       AS TOTAL_DISCOUNT,
        SUM(NET_QUANTITY)         AS NET_QUANTITY,
        MAX(IS_PLAN)              AS IS_PLAN,
        MAX(IS_GROUP)             AS IS_GROUP,
        MAX(TICKET_TYPE_CATEGORY) AS TICKET_TYPE_CATEGORY
    FROM {team}.MART.MASKED_FCT_TICKET_SALES
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE ORDER_DATE IS NOT NULL
    GROUP BY ORDER_ID
),

deduped_merch AS (
    SELECT
        ORDER_ID,
        MAX(UNIFY_ID)             AS UNIFY_ID,
        MAX(ORDER_DATE)           AS ORDER_DATE,
        SUM(TOTAL_GROSS_SALES)    AS TOTAL_GROSS_SALES,
        SUM(TOTAL_NET_SALES)      AS TOTAL_NET_SALES,
        SUM(TOTAL_DISCOUNT)       AS TOTAL_DISCOUNT,
        SUM(NET_QUANTITY)         AS NET_QUANTITY,
        MAX(PRODUCT_CATEGORY)     AS PRODUCT_CATEGORY
    FROM {team}.MART.MASKED_FCT_MERCH_TRANSACTIONS
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE ORDER_DATE IS NOT NULL
    GROUP BY ORDER_ID
),

deduped_mark AS (
    SELECT
        ACTIVITY_ID,
        MAX(UNIFY_ID)              AS UNIFY_ID,
        MAX(COALESCE(ACTIVITY_TIMESTAMP, SEND_TIMESTAMP)) AS ACTIVITY_TIMESTAMP,
        MAX(ACTION_TYPE)           AS ACTION_TYPE
    FROM {team}.MART.MASKED_FCT_MARKETING_ACTIVITIES
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE COALESCE(ACTIVITY_TIMESTAMP, SEND_TIMESTAMP) IS NOT NULL
    GROUP BY ACTIVITY_ID
),

raw_timeline AS (
    -- 1. Marketing
    SELECT
        m.UNIFY_ID,
        m.ACTIVITY_TIMESTAMP::TIMESTAMP_NTZ          AS event_timestamp,
        m.ACTION_TYPE                                 AS action_label,
        'marketing'                                   AS activity_type,
        NULL::FLOAT AS merch_total_gross_sales,
        NULL::FLOAT AS merch_total_net_sales,
        NULL::FLOAT AS merch_total_discount,
        NULL::FLOAT AS ticket_total_gross_sales,
        NULL::FLOAT AS ticket_total_net_sales,
        NULL::FLOAT AS ticket_total_discount,
        NULL::FLOAT AS merch_quantity,
        NULL::FLOAT AS ticket_quantity
    FROM deduped_mark m

    UNION ALL

    -- 2. CRM
    SELECT
        c.UNIFY_ID,
        c.ACTIVITY_TIMESTAMP::TIMESTAMP_NTZ           AS event_timestamp,
        LOWER(c.ACTIVITY_TYPE)                        AS action_label,
        'crm'                                         AS activity_type,
        NULL::FLOAT AS merch_total_gross_sales,
        NULL::FLOAT AS merch_total_net_sales,
        NULL::FLOAT AS merch_total_discount,
        NULL::FLOAT AS ticket_total_gross_sales,
        NULL::FLOAT AS ticket_total_net_sales,
        NULL::FLOAT AS ticket_total_discount,
        NULL::FLOAT AS merch_quantity,
        NULL::FLOAT AS ticket_quantity
    FROM {team}.MART.MASKED_FCT_CRM_ACTIVITIES c
    INNER JOIN sampled_users u ON c.UNIFY_ID = u.UNIFY_ID
    WHERE c.ACTIVITY_TIMESTAMP IS NOT NULL

    UNION ALL

    -- 3. Merch
    SELECT
        m.UNIFY_ID,
        m.ORDER_DATE::TIMESTAMP_NTZ                   AS event_timestamp,
        CASE
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%t-shirt%' OR LOWER(m.PRODUCT_CATEGORY) LIKE '%polo%' THEN 'tshirt'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%jersey%'                                             THEN 'jersey'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%hoodie%'                                             THEN 'hoodie'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%sweat%' OR LOWER(m.PRODUCT_CATEGORY) LIKE '%quarter-zip%' THEN 'sweatshirt'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%hat%'                                                THEN 'hat'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%ticket%'                                             THEN 'ticket'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%acces%'                                              THEN 'accessories'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%scar%'                                               THEN 'scarf'
            ELSE 'misc_merch'
        END                                           AS action_label,
        'merch'                                       AS activity_type,
        m.TOTAL_GROSS_SALES AS merch_total_gross_sales,
        m.TOTAL_NET_SALES    AS merch_total_net_sales,
        m.TOTAL_DISCOUNT     AS merch_total_discount,
        m.NET_QUANTITY       AS merch_quantity,
        NULL::FLOAT          AS ticket_total_gross_sales,
        NULL::FLOAT          AS ticket_total_net_sales,
        NULL::FLOAT          AS ticket_total_discount,
        NULL::FLOAT          AS ticket_quantity
    FROM deduped_merch m

    UNION ALL

    -- 4. Ticket Sales
    SELECT
        t.UNIFY_ID,
        t.ORDER_DATE::TIMESTAMP_NTZ                   AS event_timestamp,
        CASE
            WHEN t.IS_PLAN = 1                                           THEN 'plan'
            WHEN t.IS_GROUP = 1                                          THEN 'group'
            WHEN LOWER(t.TICKET_TYPE_CATEGORY) LIKE '%single%'          THEN 'single'
            WHEN LOWER(t.TICKET_TYPE_CATEGORY) LIKE '%season%'          THEN 'season'
            ELSE 'misc_tickets'
        END                                           AS action_label,
        'ticket'                                      AS activity_type,
        NULL::FLOAT          AS merch_total_gross_sales,
        NULL::FLOAT          AS merch_total_net_sales,
        NULL::FLOAT          AS merch_total_discount,
        NULL::FLOAT          AS merch_quantity,
        t.TOTAL_GROSS_SALES  AS ticket_total_gross_sales,
        t.TOTAL_NET_SALES    AS ticket_total_net_sales,
        t.TOTAL_DISCOUNT     AS ticket_total_discount,
        t.NET_QUANTITY       AS ticket_quantity
    FROM deduped_tickets t
),

next_ticket AS (
    SELECT
        r.*,
        MIN(CASE WHEN activity_type = 'ticket' THEN event_timestamp END)
            OVER (
                PARTITION BY UNIFY_ID
                ORDER BY event_timestamp
                ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
            ) AS next_ticket_purchase_timestamp
    FROM raw_timeline r
)

SELECT
    UNIFY_ID,
    event_timestamp                                         AS action_time,
    activity_type,
    action_label                                            AS action_taken,
    merch_total_gross_sales,
    merch_total_net_sales,
    merch_total_discount,
    merch_quantity,
    ticket_total_gross_sales,
    ticket_total_net_sales,
    ticket_total_discount,
    ticket_quantity,
    next_ticket_purchase_timestamp                          AS converted_at,
    DATEDIFF('day', event_timestamp, next_ticket_purchase_timestamp) AS days_before_purchase
FROM next_ticket
ORDER BY UNIFY_ID, action_time;
"""

In [4]:
data = query_snowflake_to_df(query)

In [5]:
data.groupby('ACTIVITY_TYPE')['ACTION_TAKEN'].value_counts()

ACTIVITY_TYPE  ACTION_TAKEN
crm            task              5677
               email             1596
               call               887
               meeting              9
marketing      open            740848
               click            35295
               subscribe         9959
               bounce            4251
               WEB_VISIT         3143
               unsubscribe       1865
merch          misc_merch         916
               scarf               66
               jersey              14
ticket         misc_tickets     17012
               plan              8618
               group             1884
               season             759
Name: count, dtype: int64

In [6]:
clean_df = data[data["ACTIVITY_TYPE"] != "crm"]

In [7]:
clean_df.groupby('ACTIVITY_TYPE')['ACTION_TAKEN'].value_counts()

ACTIVITY_TYPE  ACTION_TAKEN
marketing      open            740848
               click            35295
               subscribe         9959
               bounce            4251
               WEB_VISIT         3143
               unsubscribe       1865
merch          misc_merch         916
               scarf               66
               jersey              14
ticket         misc_tickets     17012
               plan              8618
               group             1884
               season             759
Name: count, dtype: int64

In [8]:
clean_df = clean_df[['UNIFY_ID', 'ACTIVITY_TYPE', 'ACTION_TIME', 'ACTION_TAKEN']]
clean_df.head()

,UNIFY_ID,ACTIVITY_TYPE,ACTION_TIME,ACTION_TAKEN
0,0001bd60df64bbde517d093474f542db,marketing,2024-07-28 01:09:48.000,subscribe
11,0001bd60df64bbde517d093474f542db,marketing,2026-02-21 09:56:53.000,open
12,000c4088813cac942b7c7e69e4bd36be,ticket,2018-08-27 22:59:20.917,misc_tickets
13,000c4088813cac942b7c7e69e4bd36be,marketing,2020-09-16 13:50:28.000,subscribe
14,000c4088813cac942b7c7e69e4bd36be,marketing,2020-09-26 13:25:21.000,open


## sequences

In [9]:
clean_df["ACTION_TIME"] = pd.to_datetime(clean_df["ACTION_TIME"])

# 1. Sort so sequences are in chronological order per user
df = clean_df.sort_values(["UNIFY_ID", "ACTION_TIME"], kind="stable")

# 2. Map each action to an integer (start at 1, reserve 0 for padding/unknown)
actions = sorted(df["ACTION_TAKEN"].dropna().unique())
action2id = {a: i + 1 for i, a in enumerate(actions)}
id2action = {i: a for a, i in action2id.items()}
df["ACTION_ID"] = df["ACTION_TAKEN"].map(action2id)

# 3. One sequence per user
seqs = (
    df.groupby("UNIFY_ID")["ACTION_ID"]
      .apply(list)
      .rename("ACTION_SEQ")
      .reset_index()
)
seqs["SEQ_LEN"] = seqs["ACTION_SEQ"].str.len()

In [10]:
action2id

{'WEB_VISIT': 1,
 'bounce': 2,
 'click': 3,
 'group': 4,
 'jersey': 5,
 'misc_merch': 6,
 'misc_tickets': 7,
 'open': 8,
 'plan': 9,
 'scarf': 10,
 'season': 11,
 'subscribe': 12,
 'unsubscribe': 13}

## building transformer